In [ ]:
import re
import sqlite3
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://books.toscrape.com/"
NUM_PAGES = 5

RAW_FILE = "books_raw.csv"
CLEAN_FILE = "books_clean.csv"
DATABASE_FILE = "books.db"
SQL_FILE = "sql_queries.sql"
OUTPUT_FILE = "sql_outputs.txt"
MERGE_FILE = "merge_comparison.csv"

# Required project-defined conversion rate
GBP_TO_INR = 105.50


In [3]:
def scrape_books():
    """
    Scrape the first five pages of the All Products catalogue.

    Each page contains 20 books, so the expected result is
    approximately 100 books.
    """

    books = []

    session = requests.Session()

    for page_number in range(1, NUM_PAGES + 1):

        page_url = urljoin(
            BASE_URL,
            f"catalogue/page-{page_number}.html"
        )

        print(f"Scraping page {page_number}: {page_url}")

        response = session.get(page_url, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        book_cards = soup.select("article.product_pod")

        for book in book_cards:

            # -------------------------
            # Title
            # -------------------------
            title_tag = book.select_one("h3 a")

            if title_tag is None:
                continue

            title = title_tag.get("title", "").strip()

            # -------------------------
            # Price
            # -------------------------
            price_tag = book.select_one(".price_color")

            price = (
                price_tag.get_text(strip=True)
                if price_tag
                else ""
            )

            # -------------------------
            # Star rating
            # -------------------------
            rating_tag = book.select_one("p.star-rating")

            star_rating = ""

            if rating_tag:
                classes = rating_tag.get("class", [])

                if len(classes) >= 2:
                    star_rating = classes[1]

            # -------------------------
            # Availability
            # -------------------------
            availability_tag = book.select_one(
                ".availability"
            )

            availability = (
                availability_tag.get_text(
                    " ",
                    strip=True
                )
                if availability_tag
                else ""
            )

            # -------------------------
            # Category
            # -------------------------
            relative_book_url = title_tag.get("href")

            book_url = urljoin(
                page_url,
                relative_book_url
            )

            detail_response = session.get(
                book_url,
                timeout=15
            )

            detail_response.raise_for_status()

            detail_soup = BeautifulSoup(
                detail_response.text,
                "html.parser"
            )

            breadcrumb = detail_soup.select(
                "ul.breadcrumb li"
            )

            category = ""

            # Breadcrumb:
            # Home > Books > Category > Book title
            if len(breadcrumb) >= 3:
                category = breadcrumb[2].get_text(
                    strip=True
                )

            books.append({
                "title": title,
                "price": price,
                "star_rating": star_rating,
                "availability": availability,
                "category": category
            })

    df = pd.DataFrame(books)

    print()
    print("=" * 70)
    print("SCRAPING COMPLETE")
    print("=" * 70)
    print(f"Books scraped: {len(df)}")
    print(f"Categories: {df['category'].nunique()}")

    if len(df) < 60:
        raise ValueError(
            f"Only {len(df)} books scraped. "
            "At least 60 are required."
        )

    if df["category"].nunique() < 3:
        raise ValueError(
            "At least 3 categories are required."
        )

    # Save original/raw data
    df.to_csv(
        RAW_FILE,
        index=False
    )

    print(f"Raw data saved to {RAW_FILE}")

    return df

In [5]:

# ============================================================
# 2. CLEANING
# ============================================================

def parse_price(value):
    """
    Convert a GBP price such as £51.77 to float 51.77.
    Unexpected values become NaN.
    """

    try:
        s_value = str(value)
        # Use regex to find floating point numbers. This handles various
        # non-numeric characters before/after the number more robustly.
        match = re.search(r'(\d+\.?\d*)', s_value)
        if match:
            cleaned = match.group(1)
            return float(cleaned)
        else:
            return float("nan")

    except (ValueError, TypeError):
        return float("nan")


def parse_rating(value):
    """
    Convert textual rating One-Five into integers 1-5.
    Unexpected values become NaN.
    """

    mapping = {
        "One": 1,
        "Two": 2,
        "Three": 3,
        "Four": 4,
        "Five": 5
    }

    if pd.isna(value):
        return float("nan")

    return mapping.get(
        str(value).strip(),
        float("nan")
    )


def parse_stock(value):
    """
    Convert availability text into boolean.

    In stock     -> True
    Out of stock -> False
    Unexpected    -> None
    """

    if pd.isna(value):
        return None

    text = str(value).strip().lower()

    if "in stock" in text:
        return True

    if "out of stock" in text:
        return False

    return None


def clean_books(df):
    """
    Clean scraped fields and create:
    price_gbp
    rating
    in_stock
    price_inr
    """

    df = df.copy()

    print()
    print("=" * 70)
    print("CLEANING DATA")
    print("=" * 70)

    # --------------------------------------------------------
    # Price
    # --------------------------------------------------------

    df["price_gbp"] = df["price"].apply(
        parse_price
    )

    invalid_prices = df["price_gbp"].isna().sum()

    if invalid_prices > 0:
        median_price = df["price_gbp"].median()

        print(
            f"Invalid/missing prices: {invalid_prices}"
        )

        print(
            f"Using median price: {median_price}"
        )

        df["price_gbp"] = df[
            "price_gbp"
        ].fillna(median_price)

    # --------------------------------------------------------
    # Rating
    # --------------------------------------------------------

    df["rating"] = df["star_rating"].apply(
        parse_rating
    )

    invalid_ratings = df["rating"].isna().sum()

    if invalid_ratings > 0:
        median_rating = df["rating"].median()

        print(
            f"Invalid/missing ratings: {invalid_ratings}"
        )

        print(
            f"Using median rating: {median_rating}"
        )

        df["rating"] = df[
            "rating"
        ].fillna(median_rating)

    df["rating"] = df["rating"].round().astype(int)

    # --------------------------------------------------------
    # Availability
    # --------------------------------------------------------

    df["in_stock"] = df["availability"].apply(
        parse_stock
    )

    invalid_stock = df["in_stock"].isna().sum()

    if invalid_stock > 0:

        print(
            f"Unparseable availability rows: "
            f"{invalid_stock}"
        )

        print(
            "Dropping rows with unknown stock status."
        )

        df = df.dropna(
            subset=["in_stock"]
        )

    df["in_stock"] = df[
        "in_stock"
    ].astype(bool)

    # --------------------------------------------------------
    # Clean text
    # --------------------------------------------------------

    df["title"] = (
        df["title"]
        .astype(str)
        .str.strip()
    )

    df["category"] = (
        df["category"]
        .astype(str)
        .str.strip()
    )

    # --------------------------------------------------------
    # Fixed GBP -> INR conversion
    # --------------------------------------------------------

    df["price_inr"] = (df["price_gbp"] * GBP_TO_INR).round(2)

    # --------------------------------------------------------
    # Remove duplicate books
    # --------------------------------------------------------

    before = len(df)

    df = df.drop_duplicates(
        subset=["title", "category"]
    )

    duplicates_removed = before - len(df)

    if duplicates_removed > 0:
        print(
            f"Duplicate rows removed: "
            f"{duplicates_removed}"
        )

    # --------------------------------------------------------
    # Final columns
    # --------------------------------------------------------

    df = df[
        [
            "title",
            "price_gbp",
            "rating",
            "in_stock",
            "price_inr",
            "category"
        ]
    ]

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    if not df["price_gbp"].notna().all():
        raise ValueError(
            "price_gbp still contains missing values."
        )

    if not df["rating"].between(1, 5).all():
        raise ValueError(
            "rating contains values outside 1-5."
        )

    if not df["in_stock"].notna().all():
        raise ValueError(
            "in_stock contains missing values."
        )

    if not df["price_inr"].notna().all():
        raise ValueError(
            "price_inr contains missing values."
        )

    if len(df) < 60:
        raise ValueError(
            "Cleaned dataset contains fewer than "
            "60 books."
        )

    # Save cleaned dataset
    df.to_csv(
        CLEAN_FILE,
        index=False
    )

    print()
    print("Cleaned dataset:")
    print(df.head())

    print()
    print("Data typesSPs:")
    print(df.dtypes)

    print()
    print(
        f"Cleaned books: {len(df)}"
    )

    print(
        f"Categories: {df['category'].nunique()}"
    )

    print(
        f"GBP -> INR rate: {GBP_TO_INR}"
    )

    print(
        f"Cleaned data saved to {CLEAN_FILE}"
    )

    return df

In [6]:
# ============================================================
# 3. DATABASE CREATION
# ============================================================

def create_database(df):
    """
    Create normalized SQLite database with:

    categories
        category_id PK
        category_name UNIQUE

    books
        book_id PK
        title
        price_gbp
        price_inr
        rating
        in_stock
        category_id FK
    """

    print()
    print("=" * 70)
    print("CREATING SQLITE DATABASE")
    print("=" * 70)

    conn = sqlite3.connect(
        DATABASE_FILE
    )

    # Enable foreign keys
    conn.execute(
        "PRAGMA foreign_keys = ON"
    )

    cursor = conn.cursor()

    # --------------------------------------------------------
    # Drop existing tables so the pipeline is reproducible
    # --------------------------------------------------------

    cursor.execute(
        "DROP TABLE IF EXISTS books"
    )

    cursor.execute(
        "DROP TABLE IF EXISTS categories"
    )

    # --------------------------------------------------------
    # Categories table
    # --------------------------------------------------------

    cursor.execute("""
        CREATE TABLE categories (
            category_id INTEGER PRIMARY KEY AUTOINCREMENT,
            category_name TEXT NOT NULL UNIQUE
        )
    """)

    # --------------------------------------------------------
    # Books table
    # --------------------------------------------------------

    cursor.execute("""
        CREATE TABLE books (
            book_id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT NOT NULL,
            price_gbp REAL NOT NULL,
            price_inr REAL NOT NULL,
            rating INTEGER NOT NULL,
            in_stock INTEGER NOT NULL,
            category_id INTEGER NOT NULL,

            FOREIGN KEY (category_id)
                REFERENCES categories(category_id)
        )
    """)

    # --------------------------------------------------------
    # Insert categories
    # --------------------------------------------------------

    categories = (
        df["category"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    cursor.executemany(
        """
        INSERT INTO categories (category_name)
        VALUES (?)
        """,
        [(category,) for category in categories]
    )

    # --------------------------------------------------------
    # Get category IDs
    # --------------------------------------------------------

    category_df = pd.read_sql(
        """
        SELECT category_id, category_name
        FROM categories
        """,
        conn
    )

    category_map = dict(
        zip(
            category_df["category_name"],
            category_df["category_id"]
        )
    )

    # --------------------------------------------------------
    # Insert books
    # --------------------------------------------------------

    books_to_insert = []

    for _, row in df.iterrows():

        books_to_insert.append(
            (
                row["title"],
                float(row["price_gbp"]),
                float(row["price_inr"]),
                int(row["rating"]),
                int(row["in_stock"]),
                category_map[row["category"]]
            )
        )

    cursor.executemany(
        """
        INSERT INTO books (
            title,
            price_gbp,
            price_inr,
            rating,
            in_stock,
            category_id
        )
        VALUES (?, ?, ?, ?, ?, ?)
        """,
        books_to_insert
    )

    conn.commit()

    # --------------------------------------------------------
    # Database validation
    # --------------------------------------------------------

    category_count = cursor.execute(
        "SELECT COUNT(*) FROM categories"
    ).fetchone()[0]

    book_count = cursor.execute(
        "SELECT COUNT(*) FROM books"
    ).fetchone()[0]

    print(
        f"Categories inserted: {category_count}"
    )

    print(
        f"Books inserted: {book_count}"
    )

    print(
        f"Database saved to {DATABASE_FILE}"
    )

    return conn

In [7]:
# ============================================================
# 4. SQL QUERIES
# ============================================================

SQL_QUERIES = {

    "Q1_SELECT_WHERE": """
        SELECT
            title,
            price_gbp,
            rating,
            in_stock
        FROM books
        WHERE price_gbp > 40
        ORDER BY price_gbp DESC
    """,

    "Q2_ORDER_BY_LIMIT": """
        SELECT
            title,
            price_gbp,
            rating
        FROM books
        ORDER BY price_gbp DESC
        LIMIT 10
    """,

    "Q3_DISTINCT": """
        SELECT DISTINCT
            rating
        FROM books
        ORDER BY rating
    """,

    "Q4_BETWEEN": """
        SELECT
            title,
            price_gbp,
            price_inr
        FROM books
        WHERE price_gbp BETWEEN 20 AND 40
        ORDER BY price_gbp
    """,

    "Q5_JOIN": """
        SELECT
            b.title,
            c.category_name,
            b.price_gbp,
            b.price_inr,
            b.rating,
            b.in_stock
        FROM books AS b
        INNER JOIN categories AS c
            ON b.category_id = c.category_id
        ORDER BY
            b.rating DESC,
            c.category_name,
            b.title
        LIMIT 10
    """
}


def execute_sql_queries(conn):
    """
    Execute all required SQL queries and save
    query strings + outputs.
    """

    print()
    print("=" * 70)
    print("SQL QUERIES")
    print("=" * 70)

    with open(
        SQL_FILE,
        "w",
        encoding="utf-8"
    ) as sql_file, open(
        OUTPUT_FILE,
        "w",
        encoding="utf-8"
    ) as output_file:

        for name, query in SQL_QUERIES.items():

            clean_query = query.strip()

            # Save query
            sql_file.write(
                f"-- {name}\n"
            )

            sql_file.write(
                clean_query
            )

            sql_file.write(
                "\n\n"
            )

            # Execute query
            result = pd.read_sql(
                clean_query,
                conn
            )

            # Print
            print()
            print("-" * 70)
            print(name)
            print("-" * 70)
            print(clean_query)
            print()
            print(result.to_string(index=False))

            # Save output
            output_file.write(
                "=" * 70 + "\n"
            )

            output_file.write(
                f"{name}\n"
            )

            output_file.write(
                "=" * 70 + "\n"
            )

            output_file.write(
                clean_query + "\n\n"
            )

            output_file.write(
                result.to_string(index=False)
            )

            output_file.write(
                "\n\n"
            )

In [8]:
# ============================================================
# 5. PANDAS READ_SQL + MERGE
# ============================================================

def pandas_analysis(conn):
    """
    Read SQL query results into pandas and reproduce
    the JOIN using pd.merge().
    """

    print()
    print("=" * 70)
    print("PANDAS ANALYSIS")
    print("=" * 70)

    # --------------------------------------------------------
    # Read at least two SQL results using pd.read_sql
    # --------------------------------------------------------

    df_q1 = pd.read_sql(
        SQL_QUERIES["Q1_SELECT_WHERE"],
        conn
    )

    df_q2 = pd.read_sql(
        SQL_QUERIES["Q2_ORDER_BY_LIMIT"],
        conn
    )

    df_q4 = pd.read_sql(
        SQL_QUERIES["Q4_BETWEEN"],
        conn
    )

    df_sql_join = pd.read_sql(
        SQL_QUERIES["Q5_JOIN"],
        conn
    )

    print("\nQ1 loaded using pd.read_sql:")
    print(df_q1.head())

    print("\nQ2 loaded using pd.read_sql:")
    print(df_q2)

    print("\nQ4 loaded using pd.read_sql:")
    print(df_q4.head())

    print("\nJOIN loaded using pd.read_sql:")
    print(df_sql_join)

    # --------------------------------------------------------
    # Read base tables into pandas
    # --------------------------------------------------------

    books_df = pd.read_sql(
        """
        SELECT
            book_id,
            title,
            price_gbp,
            price_inr,
            rating,
            in_stock,
            category_id
        FROM books
        """,
        conn
    )

    categories_df = pd.read_sql(
        """
        SELECT
            category_id,
            category_name
        FROM categories
        """,
        conn
    )

    # --------------------------------------------------------
    # Reproduce SQL JOIN using pd.merge
    # --------------------------------------------------------

    df_merge = pd.merge(
        books_df,
        categories_df,
        on="category_id",
        how="inner"
    )

    # Same columns as SQL JOIN
    df_merge = df_merge[
        [
            "title",
            "category_name",
            "price_gbp",
            "price_inr",
            "rating",
            "in_stock"
        ]
    ]

    # Same ordering and LIMIT as SQL
    df_merge = (
        df_merge
        .sort_values(
            by=[
                "rating",
                "category_name",
                "title"
            ],
            ascending=[
                False,
                True,
                True
            ]
        )
        .head(10)
        .reset_index(drop=True)
    )

    df_sql_compare = (
        df_sql_join
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # Compare SQL JOIN and pandas.merge
    # --------------------------------------------------------

    equivalent = df_sql_compare.equals(
        df_merge
    )

    print()
    print("=" * 70)
    print("SQL JOIN vs pandas.merge()")
    print("=" * 70)

    print("\nSQL result:")
    print(df_sql_compare)

    print("\npandas.merge() result:")
    print(df_merge)

    print(
        f"\nResults equivalent: {equivalent}"
    )

    if not equivalent:
        raise AssertionError(
            "SQL JOIN and pandas.merge() "
            "results are not equivalent."
        )

    # --------------------------------------------------------
    # Save side-by-side comparison
    # --------------------------------------------------------

    comparison = pd.DataFrame({
        "sql_join": df_sql_compare.astype(str).agg(
            " | ".join,
            axis=1
        ),
        "pandas_merge": df_merge.astype(str).agg(
            " | ".join,
            axis=1
        )
    })

    comparison["match"] = (
        comparison["sql_join"]
        == comparison["pandas_merge"]
    )

    comparison.to_csv(
        MERGE_FILE,
        index=False
    )

    print(
        f"\nComparison saved to {MERGE_FILE}"
    )

In [11]:
# ============================================================
# 6. MAIN PIPELINE
# ============================================================

def main():

    print()
    print("=" * 70)
    print("ZEPTO MODULE 1 - DATA PIPELINE")
    print("=" * 70)

    # Step 1
    raw_df = scrape_books()

    # Remove raw_df inspection debug prints
    # print("\n--- raw_df inspection ---")
    # print(raw_df.head())
    # print(raw_df.info())
    # print("-------------------------")

     #Step 2
    clean_df = clean_books(
        raw_df
    )

    # Step 3
    conn = create_database(
        clean_df
    )

    try:

        # Step 4
        execute_sql_queries(
            conn
        )

        # Step 5
        pandas_analysis(
            conn
        )

    finally:

        conn.close()

    print()
    print("=" * 70)
    print("PIPELINE COMPLETED SUCCESSFULLY")
    print("=" * 70)

    print(
        f"""
Generated files:

1. {RAW_FILE}
2. {CLEAN_FILE}
3. {DATABASE_FILE}
4. {SQL_FILE}
5. {OUTPUT_FILE}
6. {MERGE_FILE}
"""
    )


if __name__ == "__main__":
    main()


ZEPTO MODULE 1 - DATA PIPELINE


NameError: name 'requests' is not defined